# ATML PA2 - Kaggle launcher (Task 3)

Thin launcher only: clone/pull, install, get the course assets, restore earlier outputs, then call the scripts. All experiment logic lives in the `.py` files.

Notebook settings before running:

- Accelerator: **GPU T4 x2** (not P100), so the hardware matches the other runs.
- Internet: **On**.

Everything lives under `/kaggle/working/atml_PA2`. Files there are lost when an interactive session ends, so run section 12 at the end of a session and download the zip, or use Save Version > Save & Run All so Kaggle keeps `/kaggle/working` as the version's output. Every training step resumes or skips when it is already finished.

### 1. GPU check

Must show Tesla T4. The second cell keeps every script on one GPU, as on Colab.

In [ ]:
!nvidia-smi

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%env TOKENIZERS_PARALLELISM=false

### 2. Clone or update the repo

In [ ]:
%cd /kaggle/working
!test -d atml_PA2 || git clone https://github.com/Zeeba1705/atml_PA2.git
%cd /kaggle/working/atml_PA2
!git fetch origin
!git checkout task3-grpo
!git pull
!git log --oneline -3

### 3. Install the pinned requirements

In [ ]:
!pip install -q -r requirements.txt

### 4. Course assets

Downloads only when validation fails.

In [ ]:
!python -m scripts.validate_assets || (python -m scripts.download_assets && python -m scripts.validate_assets)

### 5. Restore outputs and results from an earlier session (optional)

Upload the zip from section 12 as a Kaggle Dataset and attach it to this notebook (Add Input). Kaggle unpacks it under `/kaggle/input/<dataset name>/`. This copies any `outputs/` and `results/` folders found there into the repo without overwriting newer files.

In [ ]:
%%bash
cd /kaggle/working/atml_PA2
mkdir -p outputs results

echo "attached inputs:"
find /kaggle/input -maxdepth 5 | head -40

#an upload that Kaggle left zipped is unpacked into the repo
for z in $(find /kaggle/input -name "*.zip"); do
    unzip -n -q "$z" -d . && echo "unzipped $z"
done

#an upload that Kaggle unpacked: every outputs/ or results/ folder found at any depth is copied in
for d in $(find /kaggle/input -type d \( -name outputs -o -name results \)); do
    cp -rn "$d/." "$(basename $d)/" && echo "restored $(basename $d) from $d"
done

ls outputs results

### 6. Environment check and unit tests

In [ ]:
!python -m scripts.check_environment
!python -m pytest tests -q

### 7. Quick test

2 updates of the real model from the supplied midpoint, then 8 held-out prompts. Run this before the standard run.

In [ ]:
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name quicktest --updates 2 --resume

In [ ]:
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/quicktest --name quicktest --max-examples 8

In [ ]:
!cat results/task3_grpo/quicktest/train_log.jsonl
!cat results/task3_grpo/quicktest/train_metrics.json
!cat results/task3_grpo/quicktest/eval_metrics.json

### 8. Standard GRPO continuation (20 updates)

`--resume` continues from the latest checkpoint, or reports that the run is already finished.

In [ ]:
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard --resume

In [ ]:
!tail -n 3 results/task3_grpo/standard/train_log.jsonl
!cat results/task3_grpo/standard/train_metrics.json

### 9. Group-size study

CPU only, no training, a few seconds.

In [ ]:
!python -m task3_grpo.analyze_group_size --config configs/grpo.yaml

### 10. Normalisation study

Trains and evaluates `norm_grpo` and `norm_dr_grpo` (8 updates each from the midpoint, then 200 held-out prompts each). Forks that are already finished are skipped.

In [ ]:
!python -m task3_grpo.compare_normalization --config configs/grpo.yaml

In [ ]:
!cat results/task3_grpo/normalization_study.json

### 11. Held-out evaluations of the standard run, the midpoint and the base policy

Same held-out prompts, seed, batch size and decoding as the forks. 200 prompts each. These do not skip when already done, so do not rerun a finished one.

In [ ]:
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard

In [ ]:
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter checkpoints/grpo_midpoint_policy --name midpoint

In [ ]:
!python -m task3_grpo.evaluate --config configs/grpo.yaml --name sft

### 12. Pack outputs and results for download

Writes `/kaggle/working/atml_pa2_state.zip` with `outputs/` (final adapters only, the per-update checkpoints are left out to keep it small) and `results/`. Download it from the Output panel on the right. To continue in a new session, upload it as a Dataset and attach it (section 5): finished runs are skipped, a run that was cut off mid-training starts again.

In [ ]:
%%bash
cd /kaggle/working/atml_PA2
rm -f /kaggle/working/atml_pa2_state.zip
zip -qr /kaggle/working/atml_pa2_state.zip outputs results -x "*/checkpoints/*"
ls -la /kaggle/working/atml_pa2_state.zip